# Uniprot Usage Demo
This document demonstrates how to use the Uniprot API implementation programmatically.

This api permits searching via 3 ways:
- **Stream search**: This method is suitable for text queries. It uses the same search syntax as the Uniprot website.
- **ID search**: This method is suitable for searching specific entries by their Uniprot IDs for a given pandas DataFrame.
- **Sequence search**: This method is suitable for searching entries by their protein sequences. It uses the BLAST algorithm to find similar sequences.


In [21]:
from bioseq_dl import UniprotInterface
import pandas as pd

## Stream Search

### Define the query
- Query: Should be defined as a string containing the search criteria. The query syntax is the same as the one used in the Uniprot website.
- Fields: List of entry sections to be returned. More fields can be found in the [Uniprot documentation](https://rest.uniprot.org/configure/uniprotkb/result-fields).
- Sort: Specify field by wich to sort results.

In [22]:
query="organism_name:homo sapiens (human) AND length:[15 TO 30] AND reviewed:true"
fields="accession,protein_name,sequence,ec,lineage,organism_name,ft_mutagen,ft_variant,ft_domain,ft_motif,ft_region,ft_act_site,ft_binding,ft_site,xref_pfam,xref_alphafolddb,xref_pdb,go_id"
sort="accession asc"

### Instantiating the API

In [23]:
instance = UniprotInterface(
    total_retries=5
)

In [33]:
response, _ = instance.submit_stream(
    query=query,
    fields=fields,
    sort=sort,
    include_isoform=True,
    download=False
)

### Parsing results

In [37]:
parsed, _ = instance.parse(
    results=response,
    extract_fields=None,
    format="dataframe"
)
parsed.head(5)

,accession,protein_name,organism_name,organism_id,lineage,sequence,length,alphafold_ids,biogrid_ids,brenda_ids,...,pdb_ids,pfam_ids,pride_ids,reactome_ids,refseq_ids,sabiork_ids,string_ids,active_sites,domains,variants
0,A0A075B6S0,T cell receptor gamma joining 1,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",NYYKKLFGSGTTLVVT,16,[A0A075B6S0],[],[],...,[],[],[],[],[],[],[],None,None,None
1,A0A075B6Y3,T cell receptor alpha joining 3,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",GYSSASKIIFGSGTRLSIRP,20,[A0A075B6Y3],[],[],...,[],[],[],[],[],[],[],None,None,None
2,A0A075B6Y9,T cell receptor alpha joining 42,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",YGGSQGNLIFGKGTKLSVKP,20,[A0A075B6Y9],[],[],...,[],[],[],[],[],[],[],[],"[{'type': 'Region', 'description': 'Disordered...",[]
3,A0A075B700,T cell receptor alpha joining 31,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",NNNARLMFGDGTQLVVKP,18,[A0A075B700],[],[],...,[],[],[],[],[],[],[],None,None,None
4,A0A075B706,T cell receptor delta joining 1,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",TDKLIFGKGTRVTVEP,16,[A0A075B706],[],[],...,[],[],[],[],[],[],[],None,None,None


## ID Search

### Define the query
A query has to be defined. This query should contain:
- Identifiers: A list of Uniprot IDs to be fetched.
- From db: The database from which the IDs originate. In this case, it is "UniProtKB_AC-ID".
- To db: The target database to which the IDs will be mapped. In this case,
it is "UniProtKB".

In [38]:
df = pd.DataFrame({
    "ids": ["P05067"]
})
from_db = "UniProtKB_AC-ID"
to_db = "UniProtKB"

### Instantiating the API

In [39]:
instance = UniprotInterface(
    total_retries=5
)

### Making the request

In [43]:
response, _ = instance.download_batch(
    dataset=df,
    column_ids="ids",
    auto_db=False,
    from_db=from_db,
    to_db=to_db,
    batch_size=5
)

Processing manual IDs:   0%|           0/1 [00:00<?, ?it/s] Processing manual IDs

2026-01-11 22:53:09 | INFO     | bioseq_dl.interfaces.uniprot | Fetched: 1 / 1


Processing manual IDs: 100%|██████████ 1/1 [00:03<00:00,  3.68s/it] Processing manual IDs


### Parsing results

Wether if we dont want to filter the results and get all the available fields, we can set `extract_fields=None`.

In [44]:
parsed, _ = instance.parse(response, extract_fields=None, format="dataframe")
parsed

,accession,protein_name,organism_name,gene_primary,organism_id,lineage,sequence,length,alphafold_ids,biogrid_ids,...,reactome_ids,refseq_ids,rhea_ids,sabiork_ids,string_ids,references,active_sites,domains,variants,keyword
0,P05067,Amyloid-beta precursor protein,Homo sapiens,[APP],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,770,[P05067],[106848],...,"[R-HSA-114608, R-HSA-3000178, R-HSA-381426, R-...","[NP_000475.1, NP_001129488.1, NP_001129601.1, ...",[],[P05067],[9606.ENSP00000284981],[{'title': 'The precursor of Alzheimer's disea...,"[{'type': 'Binding site', 'description': '', '...","[{'type': 'Domain', 'description': 'E1', 'loca...","[{'type': 'Natural variant', 'id': 'VAR_022315...","[3D-structure, Alternative splicing, Alzheimer..."


If we want to filter the results and get specific fields, we can set `extract_fields` to a list of desired fields. For example, to get the accession number, ID, protein name, gene names, organism name, and length, we can set `extract_fields` as follows:

In [46]:
parsed, _ = instance.parse(response, extract_fields=["accession", "id", "protein_name", "gene_names", "organism_name", "length"], format="dataframe")
parsed

,accession,protein_name,organism_name,length
0,P05067,Amyloid-beta precursor protein,Homo sapiens,770


## Sequence Search
For this search type it will need a list of sequences to be searched.

In [47]:
df = pd.read_csv("data/unknown_sequences.csv")
df

,sequence
0,MAFSDLTSRTVHLYDNWIKDADPRVEDWLLMSSPLPQTILLGFYVY...
1,MAGQHLPVPRLEGVSREQFMQHLYPQRKPLVLEGIDLGPCTSKWTV...


### Preparing BLAST
For this search type we will need to import the blast module from the bioseqdownloader package.

In [48]:
from bioseq_dl.core.utils.blast_search import (
    download_uniprot_database,
    check_blast,
    make_blast_database,
    run_blast,
    parse_blast_results
)

First we need to check if BLAST is installed in the system. If not, we need to install it.

Then, we need to download the Uniprot database in FASTA format.

In [49]:
check_blast()

2026-01-11 22:53:37 | INFO     | bioseq_dl.core.utils.blast_search | System-wide BLAST is installed.


'/home/diego/micromamba/envs/bioseqdownloader/bin/blastp'

In [50]:
download_uniprot_database("uniprotkb_reviewed", extension="fasta")
make_blast_database("uniprotkb_reviewed", extension="fasta")

2026-01-11 22:53:38 | INFO     | bioseq_dl.core.utils.blast_search | Database uniprotkb_reviewed already exists at /home/diego/.cache/bioseq_dl/blast_db/uniprotkb_reviewed.fasta.
2026-01-11 22:53:38 | INFO     | bioseq_dl.core.utils.blast_search | BLAST database already exists at /home/diego/.cache/bioseq_dl/blast_db/uniprotkb_reviewed. No need to create it again.


### Running BLAST
This will create an output file named "tmp/blast_results.txt".

In [51]:
sequences = df["sequence"].dropna().tolist()
run_blast(
    sequences=sequences,
    db_name="uniprotkb_reviewed",
    blast_type="blastp",
    evalue=1e-5,
)

2026-01-11 22:53:40 | INFO     | bioseq_dl.core.utils.blast_search | Running BLAST search...


After that, we can parse the results and get a pandas DataFrame with the results.

In [52]:
results = parse_blast_results("tmp/blast_results.txt")
results_df = pd.DataFrame(results)
results_df.head()

,query,subject,identity,alignment_length,evalue,bit_score,coverage
0,0,sp|A1L3X0|ELOV7_HUMAN,100.000,281,0.0,583,100
1,0,sp|A0JNC4|ELOV7_BOVIN,91.103,281,0.0,535,100
2,1,sp|A2RUC4|TYW5_HUMAN,100.000,315,0.0,654,100


Now you can manipulate the DataFrame as needed. For example, you can rename columns, drop unnecessary columns, and extract specific information from the subject IDs.

In [53]:
df_blast = results_df.rename(columns={"query": "id", "subject": "subject_id"})
df_blast = df_blast.drop(columns=["id"])
df_blast["accession"] = df_blast["subject_id"].apply(lambda x: x.split("|")[1])
df_blast = df_blast.drop(columns=["subject_id","alignment_length", "evalue", "bit_score", "coverage"])
df_blast

,identity,accession
0,100.000,A1L3X0
1,91.103,A0JNC4
2,100.000,A2RUC4


After done, you can do a search in the Uniprot database using the accession numbers obtained from the BLAST results to get more information about the sequences.

In [54]:
instance = UniprotInterface()
results, _ = instance.download_batch(
    dataset=df_blast,
    column_ids="accession",
    auto_db=False,
    from_db="UniProtKB_AC-ID",
    to_db="UniProtKB",
    batch_size=10
)

Processing manual IDs:   0%|           0/3 [00:00<?, ?it/s] Processing manual IDs

2026-01-11 22:53:48 | INFO     | bioseq_dl.interfaces.uniprot | Fetched: 3 / 3


Processing manual IDs: 100%|██████████ 3/3 [00:03<00:00,  1.09s/it] Processing manual IDs


Parsing results

In [55]:
final_df, _ = instance.parse(results, extract_fields=["accession", "id", "protein_name", "sequence", "gene_names", "organism_name", "length"], format="dataframe")
final_df

,accession,protein_name,organism_name,sequence,length
0,A1L3X0,Very long chain fatty acid elongase 7,Homo sapiens,MAFSDLTSRTVHLYDNWIKDADPRVEDWLLMSSPLPQTILLGFYVY...,281
1,A0JNC4,Very long chain fatty acid elongase 7,Bos taurus,MAFSDLTSRTVRLYDNWIKDADPRVEDWLLMSSPLPQTIILGFYVY...,281
2,A2RUC4,tRNA wybutosine-synthesizing protein 5,Homo sapiens,MAGQHLPVPRLEGVSREQFMQHLYPQRKPLVLEGIDLGPCTSKWTV...,315


# Data Enrichement
Furthermore, you can enrich your data by performing additional searches based on a given identifiers. For example, you can search antibacterial activity and return their structures by using alphafold and PDB.

In [56]:
from bioseq_dl import UniprotInterface

In [58]:
#query = "antibacterial AND reviewed:true"
query = "B0L3A2 OR Q69383"
fields="accession,protein_name,sequence,ec,lineage,organism_name,go,xref_brenda,xref_alphafolddb"
sort="accession asc"

### Instantiating the API

In [59]:
instance = UniprotInterface()

### Making a request

In [60]:
response_data, search_metadata = instance.submit_stream(
    query=query,
    fields=fields,
    sort=sort,
    include_isoform=True,
    download=False,
)

In [61]:
parsed_data, parse_metadata = instance.parse(
    results=response_data,
    extract_fields=None,
    format="dataframe"
)

Once parsed results are obtained you can use the CrossRefEnricher class to enrich your data based on a given field.

In [62]:
from bioseq_dl.core.crossref_enricher import CrossRefEnricher, EndpointSpec

First you have to define the specific enricher method you want to use. For example to define an Alphafold and PDB enricher:

In [63]:
specs = [
    EndpointSpec(database="alphafold", endpoint="prediction", option=None, params={}),
    #EndpointSpec(database="pdb", endpoint="entry", option=None, params={}),
    #EndpointSpec(database="brenda", endpoint="getKmValue", option=None, params={})
]

### Instantating the enricher

In [64]:
enricher = CrossRefEnricher(specs)

### Using the enricher
For this particular example we will enrich only the first 40 entries to avoid long processing times.

The following parameters are available:
- data: The data to be enriched. It can be a pandas DataFrame or a list of dictionaries. Normally this comes from a previous uniprot search.
- concat_results: If True, the results of the enrichement will be concatenated alongside the row being processed. For example if a row contains an identifier that maps to 3 entries in the target database, the resulting DataFrame will contain 3 rows for that original row, each one containing the data from the original row plus the data from each of the 3 mapped entries. If False, the results will be returned as a list of dictionaries, where each dictionary contains the search data without concatenating the original row.
- to_dataframe: If True, the results will be returned as a pandas DataFrame. If False, the results will be returned as a list of dictionaries.

The returned values are:
- enriched_data: The enriched data. It can be a pandas DataFrame or a list of dictionaries depending on the value of the `to_dataframe` parameter.
- enriched_metadata: A dictionary containing metadata about the enrichment process, such as the number of successful and failed searches.

In [67]:
enriched_data, enriched_metadata = enricher.enrich(parsed_data, format="dataframe")
enriched_data["alphafold_prediction"]

2026-01-11 22:55:50 | INFO     | bioseq_dl.interfaces.crossref_enricher | Checking availability for interface: alphafold
2026-01-11 22:55:50 | INFO     | bioseq_dl.interfaces.crossref_enricher | Checking required columns for alphafold:prediction...
2026-01-11 22:55:50 | INFO     | bioseq_dl.interfaces.crossref_enricher | Building interface for alphafold...
2026-01-11 22:55:50 | INFO     | bioseq_dl.interfaces.crossref_enricher | Prepared params for alphafold:prediction: {}
2026-01-11 22:55:50 | INFO     | bioseq_dl.interfaces.alphafold | Structure AF-B0L3A2-F1-model_v6.pdb already exists. Skipping download.
2026-01-11 22:55:50 | INFO     | bioseq_dl.interfaces.alphafold | Structure AF-Q69383-F1-model_v6.pdb already exists. Skipping download.


,entry,gene,tax_id,organism,is_reviewed,is_reference,pdbUrl
0,AF-B0L3A2-F1,FBXW7-AS1,9606,Homo sapiens,True,True,https://alphafold.ebi.ac.uk/files/AF-B0L3A2-F1...
1,AF-Q69383-F1,ERVK-6,9606,Homo sapiens,True,True,https://alphafold.ebi.ac.uk/files/AF-Q69383-F1...
